In [ ]:
import pandas as pd

model_df = pd.read_csv(
    "../data_clean/leishmania_ml_dataset.csv"
)

model_df.head()

,Molecule ChEMBL ID,SMILES,pIC50,Molecular_Weight,AlogP,TPSA,HBA,HBD,RO5_Violations,Rotatable_Bonds,QED,Aromatic_Rings,Heavy_Atoms,NP_Likeness
0,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,3.940555,388.90,3.15,53.01,4.0,1.0,0.0,8.0,0.70,2.0,27.0,-1.21
1,CHEMBL100210,CCCC[C@H]1C(=O)O[C@@H]2O[C@@]3(CC)CC[C@H]4[C@H...,4.443697,338.44,3.96,53.99,5.0,0.0,0.0,4.0,0.57,0.0,24.0,2.52
2,CHEMBL100740,C[C@@H]1CC[C@H]2[C@@H](C)C(=O)O[C@@H]3OC(C)(C)...,3.800519,284.35,2.64,53.99,5.0,0.0,0.0,0.0,0.51,0.0,20.0,2.66
3,CHEMBL104,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,5.242633,344.85,5.38,17.82,2.0,0.0,1.0,4.0,0.45,4.0,25.0,-0.58
4,CHEMBL106,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,3.898138,306.28,0.74,81.65,7.0,1.0,0.0,5.0,0.75,3.0,22.0,-0.86


In [2]:
from rdkit import Chem

model_df["Mol"] = model_df["SMILES"].apply(
    Chem.MolFromSmiles
)

print(model_df["Mol"].notnull().sum())

5457


In [3]:
from rdkit.Chem import AllChem

model_df["FP"] = model_df["Mol"].apply(
    lambda mol:
    AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=1024
    )
)

[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerator
[17:06:40] DEPRECATION WARNING: please use MorganGenerat

In [4]:
subset = model_df.iloc[:500].copy()

In [ ]:
from rdkit.DataStructs import TanimotoSimilarity

activity_cliffs = []

for i in range(len(subset)):

    fp1 = subset.iloc[i]["FP"]
    p1 = subset.iloc[i]["pIC50"]

    for j in range(i+1, len(subset)):

        fp2 = subset.iloc[j]["FP"]
        p2 = subset.iloc[j]["pIC50"]

        similarity = TanimotoSimilarity(
            fp1,
            fp2
        )

        delta_activity = abs(p1 - p2)

        if (
            similarity >= 0.80
            and
            delta_activity >= 2
        ):

            activity_cliffs.append([
                subset.iloc[i]["Molecule ChEMBL ID"],
                subset.iloc[j]["Molecule ChEMBL ID"],
                similarity,
                delta_activity
            ])